# TikTok GraphRAG Embedding Pipeline (production)

This notebook:

- Embeds `TikTokVideo.comment_summary_description` → `comment_summary_embedding`
- Embeds combined video text → `video_content_embedding`
- **Syncs comment topics from MongoDB** (`tiktok_user_video`) into Neo4j as `(:TikTokCommentTopic)` and `[:HAS_COMMENT_TOPIC]`, **only** when `(:TikTokVideo {video_id})` already exists with the **same `video_id` string** as Mongo.
- Embeds `TikTokCommentTopic.name` → `embedding` and ensures a dedicated vector index.

Uses Neo4j / Mongo / embedding credentials from `.env`. Re-runs are idempotent.
> **Run order:** After setup and the **Video comment summary embedding functions** cell, scroll to the heading **Run comment summary embeddings (embedding pass)** and run the code cell below it (`# === COMMENT SUMMARY EMBEDDING PASS ===`). Then **video content embedding**, then **topic sync + topic embeddings** (immediately before `driver.close()`).


In [10]:
import json
import os
import time
from typing import Any, Dict, Iterable, Iterator, List, Optional

from dotenv import load_dotenv
from neo4j import GraphDatabase
from openai import AzureOpenAI, OpenAI
from pymongo import MongoClient

load_dotenv()
print('Env loaded')



Env loaded


In [11]:
def _redact_bolt_uri(uri: str) -> str:
    if not uri:
        return ''
    if '@' not in uri:
        return uri
    left, right = uri.rsplit('@', 1)
    if '://' in left:
        scheme = left.split('://', 1)[0]
        return f'{scheme}://***:***@{right}'
    return uri


def _redact_mongo_uri(uri: str) -> str:
    if not uri or '://' not in uri:
        return uri
    scheme, rest = uri.split('://', 1)
    if '@' not in rest:
        return uri
    _creds, hostpart = rest.rsplit('@', 1)
    return f'{scheme}://***:***@{hostpart}'


# Production Neo4j from .env
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USER = os.getenv('NEO4J_USER')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

# Mongo (same collection as DAG / tiktok_embedding_test)
MONGO_URI = os.getenv('MONGODB_URI') or os.getenv('MONGO_URI')
MONGO_DB = os.getenv('MONGODB_DB') or os.getenv('MONGO_DB', 'rbl')
MONGO_COLLECTION = os.getenv('MONGO_COLLECTION', 'tiktok_user_video')

# Embeddings config from .env
AZURE_OPENAI_EMBEDDING_ENDPOINT = os.getenv('AZURE_OPENAI_EMBEDDING_ENDPOINT')
AZURE_OPENAI_EMBEDDING_API_KEY = os.getenv('AZURE_OPENAI_EMBEDDING_API_KEY')
AZURE_OPENAI_EMBEDDING_API_VERSION = os.getenv('AZURE_OPENAI_EMBEDDING_API_VERSION', '2024-02-01')
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = (
    os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT_MODEL_NAME')
    or os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME')
    or os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')
    or 'text-embedding-3-large'
)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_EMBEDDING_MODEL = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-large')

# Tuning knobs
PAGE_SIZE = int(os.getenv('TT_SUMMARY_EMBED_PAGE_SIZE', '500'))
EMBED_BATCH = int(os.getenv('TT_SUMMARY_EMBED_BATCH', '128'))
WRITE_BATCH = int(os.getenv('TT_SUMMARY_WRITE_BATCH', '500'))  # legacy name; vector writes use *_VECTOR_WRITE_BATCH
SUMMARY_VECTOR_WRITE_BATCH = int(os.getenv('TT_SUMMARY_VECTOR_WRITE_BATCH', '32'))  # small txs for Aura (timeout-safe)
CONTENT_VECTOR_WRITE_BATCH = int(os.getenv('TT_CONTENT_VECTOR_WRITE_BATCH', '32'))
NEO4J_CONNECTION_TIMEOUT = float(os.getenv('NEO4J_CONNECTION_TIMEOUT', '120'))  # seconds (Bolt TCP/TLS)
TOPIC_UPSERT_BATCH = int(os.getenv('TT_TOPIC_UPSERT_BATCH', '200'))
TOPIC_SYNC_LIMIT = int(os.getenv('TT_TOPIC_SYNC_LIMIT', '0'))  # 0 = scan full collection
TOPIC_EMBED_WRITE_BATCH = int(os.getenv('TT_TOPIC_EMBED_WRITE_BATCH', '32'))  # small txs for topic vector writes (Aura / prod)

TIKTOK_PLATFORM = 'tiktok'

assert NEO4J_URI and NEO4J_USER and NEO4J_PASSWORD, 'Set NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD in .env'
assert MONGO_URI, 'Set MONGODB_URI or MONGO_URI in .env for topic sync'

print('Config:')
print(f'  neo4j: {_redact_bolt_uri(NEO4J_URI)}  db={NEO4J_DATABASE}  user={NEO4J_USER}')
print(f'  mongo: {_redact_mongo_uri(MONGO_URI)}  db={MONGO_DB}  collection={MONGO_COLLECTION}')
print(f'  batches: page={PAGE_SIZE} embed={EMBED_BATCH} write={WRITE_BATCH} summary_vec={SUMMARY_VECTOR_WRITE_BATCH} content_vec={CONTENT_VECTOR_WRITE_BATCH} topic_upsert={TOPIC_UPSERT_BATCH}  topic_sync_limit={TOPIC_SYNC_LIMIT or "none"}  topic_vec_write={TOPIC_EMBED_WRITE_BATCH}  neo4j_conn_timeout={NEO4J_CONNECTION_TIMEOUT}s')



Config:
  neo4j: bolt+ssc://rbl-neo4j.ecda.ai:7687  db=neo4j  user=neo4j
  mongo: mongodb://***:***@  db=rbl  collection=tiktok_user_video
  batches: page=500 embed=128 write=500 summary_vec=32 content_vec=32 topic_upsert=200  topic_sync_limit=none  topic_vec_write=32  neo4j_conn_timeout=120.0s


In [12]:
def build_embedding_client():
    if AZURE_OPENAI_EMBEDDING_ENDPOINT and AZURE_OPENAI_EMBEDDING_API_KEY:
        client = AzureOpenAI(
            azure_endpoint=AZURE_OPENAI_EMBEDDING_ENDPOINT,
            api_key=AZURE_OPENAI_EMBEDDING_API_KEY,
            api_version=AZURE_OPENAI_EMBEDDING_API_VERSION,
        )
        return 'azure', client, AZURE_OPENAI_EMBEDDING_DEPLOYMENT
    if OPENAI_API_KEY:
        return 'openai', OpenAI(api_key=OPENAI_API_KEY), OPENAI_EMBEDDING_MODEL
    raise RuntimeError('No embedding credentials found in .env')


def batched(iterable: Iterable, n: int) -> Iterator[list]:
    bucket = []
    for item in iterable:
        bucket.append(item)
        if len(bucket) >= n:
            yield bucket
            bucket = []
    if bucket:
        yield bucket


def _video_id_str(raw_id: Any) -> str:
    if raw_id is None:
        return ''
    if isinstance(raw_id, dict) and '$numberLong' in raw_id:
        return str(raw_id['$numberLong']).strip()
    return str(raw_id).strip()


EMBED_PROVIDER, embed_client, EMBED_MODEL_NAME = build_embedding_client()
probe = embed_client.embeddings.create(model=EMBED_MODEL_NAME, input=['ping'])
EMBED_DIM = len(probe.data[0].embedding)

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
    connection_timeout=NEO4J_CONNECTION_TIMEOUT,
)
with driver.session(database=NEO4J_DATABASE) as s:
    print('Neo4j ok:', s.run('RETURN 1 AS ok').single()['ok'])
    pending = s.run(
        'MATCH (v:TikTokVideo) WHERE v.comment_summary_description IS NOT NULL AND v.comment_summary_embedding IS NULL RETURN count(v) AS c'
    ).single()['c']
    print('Pending TikTok summary embeddings:', pending)

mongo_client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=30_000)
mongo_coll = mongo_client[MONGO_DB][MONGO_COLLECTION]
mongo_client.admin.command('ping')
print('Mongo ok:', MONGO_DB, MONGO_COLLECTION)

print(f'Embedding provider: {EMBED_PROVIDER}  model/deployment: {EMBED_MODEL_NAME}  dim={EMBED_DIM}')


Neo4j ok: 1
Pending TikTok summary embeddings: 0


/tmp/ipykernel_31531/2650715156.py:49: UserWarning: You appear to be connected to a CosmosDB cluster. For more information regarding feature compatibility and support please visit https://www.mongodb.com/supportability/cosmosdb
  mongo_client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=30_000)


Mongo ok: rbl tiktok_user_video
Embedding provider: azure  model/deployment: text-embedding-3-large  dim=3072


## Video comment summary embedding functions

Defines summary fetch/write Cypher and the `embed_tiktok_comment_summaries` runner.

In [13]:

def _write_summary_vector_chunk(driver, rows_chunk):
    """Small Neo4j transactions + retries (Aura / slow networks)."""
    from neo4j.exceptions import Neo4jError, ServiceUnavailable

    attempt = 0
    while True:
        try:
            with driver.session(database=NEO4J_DATABASE) as s:
                rec = s.run(WRITE_SUMMARY_CYPHER, rows=rows_chunk).single()
                return int(rec['written']) if rec else 0
        except (Neo4jError, ServiceUnavailable, TimeoutError, OSError) as e:
            attempt += 1
            if attempt > 8:
                raise
            wait = min(2 ** attempt, 120)
            print(f'  neo4j summary-vector write retry {attempt} after {wait}s: {e}')
            time.sleep(wait)


def _write_content_vector_chunk(driver, rows_chunk):
    from neo4j.exceptions import Neo4jError, ServiceUnavailable

    attempt = 0
    while True:
        try:
            with driver.session(database=NEO4J_DATABASE) as s:
                rec = s.run(WRITE_CONTENT_CYPHER, rows=rows_chunk).single()
                return int(rec['written']) if rec else 0
        except (Neo4jError, ServiceUnavailable, TimeoutError, OSError) as e:
            attempt += 1
            if attempt > 8:
                raise
            wait = min(2 ** attempt, 120)
            print(f'  neo4j content-vector write retry {attempt} after {wait}s: {e}')
            time.sleep(wait)

def ensure_vector_indexes(driver, dim: int) -> None:
    summary_stmt = f'''CREATE VECTOR INDEX tiktok_video_summary_embedding_index IF NOT EXISTS
    FOR (v:TikTokVideo) ON (v.comment_summary_embedding)
    OPTIONS {{ indexConfig: {{ `vector.dimensions`: {dim}, `vector.similarity_function`: 'cosine' }} }}'''
    content_stmt = f'''CREATE VECTOR INDEX tiktok_video_content_embedding_index IF NOT EXISTS
    FOR (v:TikTokVideo) ON (v.video_content_embedding)
    OPTIONS {{ indexConfig: {{ `vector.dimensions`: {dim}, `vector.similarity_function`: 'cosine' }} }}'''
    with driver.session(database=NEO4J_DATABASE) as s:
        s.run(summary_stmt)
        s.run(content_stmt)
    print('Ensured tiktok_video_summary_embedding_index and tiktok_video_content_embedding_index')


def embed_texts(texts: List[str], max_retries: int = 5) -> List[List[float]]:
    out: List[List[float]] = []
    for chunk in batched(texts, EMBED_BATCH):
        attempt = 0
        while True:
            try:
                resp = embed_client.embeddings.create(model=EMBED_MODEL_NAME, input=chunk)
                out.extend([d.embedding for d in resp.data])
                break
            except Exception as e:
                attempt += 1
                if attempt > max_retries:
                    raise
                wait = min(2 ** attempt, 30)
                print(f'  embed retry {attempt} after {wait}s: {e}')
                time.sleep(wait)
    return out


FETCH_SUMMARY_CYPHER = '''
MATCH (v:TikTokVideo)
WHERE v.comment_summary_description IS NOT NULL
  AND v.comment_summary_embedding IS NULL
RETURN elementId(v) AS eid, v.comment_summary_description AS text
LIMIT $limit
'''

WRITE_SUMMARY_CYPHER = '''
UNWIND $rows AS row
MATCH (v) WHERE elementId(v) = row.eid
CALL db.create.setNodeVectorProperty(v, 'comment_summary_embedding', row.embedding)
RETURN count(*) AS written
'''

WRITE_CONTENT_CYPHER = '''
UNWIND $rows AS row
MATCH (v) WHERE elementId(v) = row.eid
SET v.video_content_text = row.text
WITH v, row
CALL db.create.setNodeVectorProperty(v, 'video_content_embedding', row.embedding)
RETURN count(*) AS written
'''


def embed_tiktok_comment_summaries(driver, page_size: int = PAGE_SIZE) -> int:
    total_written = 0
    start = time.time()
    while True:
        with driver.session(database=NEO4J_DATABASE) as s:
            pending = list(s.run(FETCH_SUMMARY_CYPHER, limit=page_size))
        if not pending:
            break

        texts = [r['text'] for r in pending]
        embeddings = embed_texts(texts)
        rows = [{'eid': pending[i]['eid'], 'embedding': embeddings[i]} for i in range(len(pending))]

        for chunk in batched(rows, SUMMARY_VECTOR_WRITE_BATCH):
            written = _write_summary_vector_chunk(driver, chunk)
            total_written += written

        print(f'  summary written={total_written}  elapsed={time.time()-start:.1f}s')

    return total_written


In [ ]:
# Summary status check (under comment summary section)
with driver.session(database=NEO4J_DATABASE) as s:
    summary_totals = {
        'videos_with_summary': s.run('MATCH (v:TikTokVideo) WHERE v.comment_summary_description IS NOT NULL RETURN count(v) AS c').single()['c'],
        'videos_with_summary_embedding': s.run('MATCH (v:TikTokVideo) WHERE v.comment_summary_embedding IS NOT NULL RETURN count(v) AS c').single()['c'],
        'summary_pending': s.run('MATCH (v:TikTokVideo) WHERE v.comment_summary_description IS NOT NULL AND v.comment_summary_embedding IS NULL RETURN count(v) AS c').single()['c'],
    }

print(summary_totals)


{'videos_with_summary': 17325, 'videos_with_summary_embedding': 17325, 'summary_pending': 0}


## Run comment summary embeddings (embedding pass)

Run the **next code cell** to populate `comment_summary_embedding`.


In [14]:
# === COMMENT SUMMARY EMBEDDING PASS ===
ensure_vector_indexes(driver, EMBED_DIM)
summary_written = embed_tiktok_comment_summaries(driver)
print('Done. Newly embedded TikTok comment summaries:', summary_written)

with driver.session(database=NEO4J_DATABASE) as s:
    summary_totals = {
        'videos_with_summary': s.run('MATCH (v:TikTokVideo) WHERE v.comment_summary_description IS NOT NULL RETURN count(v) AS c').single()['c'],
        'videos_with_summary_embedding': s.run('MATCH (v:TikTokVideo) WHERE v.comment_summary_embedding IS NOT NULL RETURN count(v) AS c').single()['c'],
        'summary_pending': s.run('MATCH (v:TikTokVideo) WHERE v.comment_summary_description IS NOT NULL AND v.comment_summary_embedding IS NULL RETURN count(v) AS c').single()['c'],
    }
print(summary_totals)


Ensured tiktok_video_summary_embedding_index and tiktok_video_content_embedding_index
Done. Newly embedded TikTok comment summaries: 0
{'videos_with_summary': 17325, 'videos_with_summary_embedding': 17325, 'summary_pending': 0}


## Video content embedding functions

Defines helper text construction, content fetch/write Cypher, and the `embed_tiktok_video_content` runner.

In [15]:
# TikTok video content embedding helpers and cypher

def format_sticker_info_list(val: Any) -> Optional[str]:
    if val is None:
        return None
    if isinstance(val, str):
        return val.strip() or None
    if isinstance(val, list) and len(val) == 0:
        return None
    try:
        return json.dumps(val, ensure_ascii=False)
    except (TypeError, ValueError):
        return str(val)


def build_tiktok_video_content_text(
    title: Optional[str],
    thumbnail_description: Optional[str],
    thumbnail_keywords: List[str],
    hashtag_names: List[str],
    description: Optional[str],
    voice_to_text: Optional[str],
    sticker_info_list: Any,
) -> Optional[str]:
    parts: List[str] = []
    if title:
        parts.append(f'Title: {title}')
    if thumbnail_description:
        parts.append(f'Thumbnail: {thumbnail_description}')
    if thumbnail_keywords:
        parts.append(f'Thumbnail keywords: {", ".join(thumbnail_keywords)}')
    if hashtag_names:
        parts.append(f'Hashtags: {", ".join(hashtag_names)}')
    if description:
        parts.append(f'Description: {description}')
    if voice_to_text:
        parts.append(f'Voice to text: {voice_to_text}')
    sticker_str = format_sticker_info_list(sticker_info_list)
    if sticker_str:
        parts.append(f'Stickers: {sticker_str}')
    return '\n'.join(parts).strip() if parts else None


FETCH_CONTENT_CYPHER = '''
MATCH (v:TikTokVideo)
WHERE v.video_content_embedding IS NULL
  AND (
    v.video_title IS NOT NULL
    OR v.video_thumbnail_description IS NOT NULL
    OR size(coalesce(v.video_thumbnail_keywords, [])) > 0
    OR EXISTS { (v)-[:HAS_TAG]->(:TikTokHashTag) }
    OR v.video_description IS NOT NULL
    OR v.voice_to_text IS NOT NULL
    OR EXISTS { (v)-[:HAS_STICKER]->(:TikTokSticker) }
  )
OPTIONAL MATCH (v)-[:HAS_TAG]->(ht:TikTokHashTag)
WITH v,
     elementId(v) AS eid,
     v.video_title AS video_title,
     v.video_thumbnail_description AS video_thumbnail_description,
     coalesce(v.video_thumbnail_keywords, []) AS video_thumbnail_keywords,
     v.video_description AS video_description,
     v.voice_to_text AS voice_to_text,
     [x IN collect(DISTINCT ht.name) WHERE x IS NOT NULL] AS hashtag_names_from_rels
OPTIONAL MATCH (v)-[:HAS_STICKER]->(st:TikTokSticker)
WITH eid,
     video_title,
     video_thumbnail_description,
     video_thumbnail_keywords,
     video_description,
     voice_to_text,
     hashtag_names_from_rels,
     [x IN collect(DISTINCT st.name) WHERE x IS NOT NULL] AS sticker_names_from_rels
RETURN eid,
       video_title,
       video_thumbnail_description,
       video_thumbnail_keywords,
       hashtag_names_from_rels,
       video_description,
       voice_to_text,
       sticker_names_from_rels
LIMIT $limit
'''


def embed_tiktok_video_content(driver, page_size: int = PAGE_SIZE) -> int:
    total_written = 0
    start = time.time()
    while True:
        with driver.session(database=NEO4J_DATABASE) as s:
            pending = list(s.run(FETCH_CONTENT_CYPHER, limit=page_size))
        if not pending:
            break

        rows_to_embed = []
        texts = []
        for record in pending:
            hn = record['hashtag_names_from_rels'] or []
            hashtag_names = [h for h in hn if isinstance(h, str) and h]
            sn = record['sticker_names_from_rels'] or []
            sticker_payload = [x for x in sn if isinstance(x, str) and x] or None
            tk = [k for k in (record['video_thumbnail_keywords'] or []) if isinstance(k, str) and k]
            text = build_tiktok_video_content_text(
                record['video_title'],
                record['video_thumbnail_description'],
                tk,
                hashtag_names,
                record['video_description'],
                record['voice_to_text'],
                sticker_payload,
            )
            if text:
                rows_to_embed.append({'eid': record['eid'], 'text': text})
                texts.append(text)

        if not rows_to_embed:
            break

        embeddings = embed_texts(texts)
        write_rows = [
            {'eid': rows_to_embed[i]['eid'], 'text': rows_to_embed[i]['text'], 'embedding': embeddings[i]}
            for i in range(len(rows_to_embed))
        ]

        for chunk in batched(write_rows, CONTENT_VECTOR_WRITE_BATCH):
            total_written += _write_content_vector_chunk(driver, chunk)

        print(f'  content written={total_written}  elapsed={time.time()-start:.1f}s')

    return total_written


## Run video content embedding

Runs only `video_content_embedding` creation and reports content embedding totals.

In [ ]:
# Run video content embeddings only (comment summary already completed)
with driver.session(database=NEO4J_DATABASE) as s:
    s.run(
        f'''CREATE VECTOR INDEX tiktok_video_content_embedding_index IF NOT EXISTS
        FOR (v:TikTokVideo) ON (v.video_content_embedding)
        OPTIONS {{ indexConfig: {{ `vector.dimensions`: {EMBED_DIM}, `vector.similarity_function`: 'cosine' }} }}'''
    )

content_written = embed_tiktok_video_content(driver)
print('Done. Newly embedded TikTok video content:', content_written)

with driver.session(database=NEO4J_DATABASE) as s:
    content_totals = {
        'videos_with_content_embedding': s.run('MATCH (v:TikTokVideo) WHERE v.video_content_embedding IS NOT NULL RETURN count(v) AS c').single()['c'],
        'content_pending': s.run("MATCH (v:TikTokVideo) WHERE v.video_content_embedding IS NULL AND (v.video_title IS NOT NULL OR v.video_thumbnail_description IS NOT NULL OR size(coalesce(v.video_thumbnail_keywords, [])) > 0 OR EXISTS { (v)-[:HAS_TAG]->(:TikTokHashTag) } OR v.video_description IS NOT NULL OR v.voice_to_text IS NOT NULL OR EXISTS { (v)-[:HAS_STICKER]->(:TikTokSticker) }) RETURN count(v) AS c").single()['c'],
    }

print(content_totals)


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `sticker_info_list` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=10, column=10, offset=280>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 280, 'line': 10, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (v:TikTokVideo)\nWHERE v.video_content_embedding IS NULL\n  AND (\n    v.video_title IS NOT NULL\n    OR v.video_thumbnail_description IS NOT NULL\n    OR size(coalesce(v.hashtag_names, [])) > 0\n    OR v.video_description IS NOT NULL\n    OR v.voice_to_text IS NOT NULL\n    OR v.sticker_info_list IS NOT NULL\n 

  content written=498  elapsed=42.8s


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `sticker_info_list` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=10, column=10, offset=280>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 280, 'line': 10, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (v:TikTokVideo)\nWHERE v.video_content_embedding IS NULL\n  AND (\n    v.video_title IS NOT NULL\n    OR v.video_thumbnail_description IS NOT NULL\n    OR size(coalesce(v.hashtag_names, [])) > 0\n    OR v.video_description IS NOT NULL\n    OR v.voice_to_text IS NOT NULL\n    OR v.sticker_info_list IS NOT NULL\n 

  content written=996  elapsed=84.2s


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `sticker_info_list` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=10, column=10, offset=280>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 280, 'line': 10, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (v:TikTokVideo)\nWHERE v.video_content_embedding IS NULL\n  AND (\n    v.video_title IS NOT NULL\n    OR v.video_thumbnail_description IS NOT NULL\n    OR size(coalesce(v.hashtag_names, [])) > 0\n    OR v.video_description IS NOT NULL\n    OR v.voice_to_text IS NOT NULL\n    OR v.sticker_info_list IS NOT NULL\n 

  content written=1491  elapsed=124.7s


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `sticker_info_list` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=10, column=10, offset=280>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 280, 'line': 10, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (v:TikTokVideo)\nWHERE v.video_content_embedding IS NULL\n  AND (\n    v.video_title IS NOT NULL\n    OR v.video_thumbnail_description IS NOT NULL\n    OR size(coalesce(v.hashtag_names, [])) > 0\n    OR v.video_description IS NOT NULL\n    OR v.voice_to_text IS NOT NULL\n    OR v.sticker_info_list IS NOT NULL\n 

## Comment topics — Mongo -> Neo4j (`TikTokCommentTopic`)

Reads `comments_frequent_topics` (+ categories / weights) from `tiktok_user_video`.
For each document, only if `(:TikTokVideo {video_id})` exists in Neo4j with the same numeric `video_id` as Mongo, existing `HAS_COMMENT_TOPIC` relationships from that video are removed and recreated pointing at `(:TikTokCommentTopic {video_id, name})`.

Videos that exist in Mongo but not yet in Neo4j are skipped (no orphan topic relationships).


In [16]:
TIKTOK_TOPICS_MONGO_PROJECTION = {
    'video_id': 1,
    'comments_frequent_topics': 1,
    'comments_frequent_topic_categories': 1,
    'comments_frequent_topic_weights': 1,
}


def clean_text(value: Any) -> Optional[str]:
    if value is None:
        return None
    if isinstance(value, str):
        return value
    return str(value)


def topic_row_from_mongo_doc(doc: dict) -> Optional[Dict[str, Any]]:
    """One Mongo doc → {video_id: int, topics: [...]} — same fields as tiktok_embedding_test.normalize_mongo_doc topics slice."""
    raw_id = doc.get('video_id')
    if raw_id is None:
        return None
    if isinstance(raw_id, dict) and '$numberLong' in raw_id:
        raw_id = raw_id['$numberLong']
    try:
        video_id = int(str(raw_id).strip())
    except (TypeError, ValueError):
        return None
    names = doc.get('comments_frequent_topics') or []
    categories = doc.get('comments_frequent_topic_categories') or []
    weights = doc.get('comments_frequent_topic_weights') or []
    topics: List[Dict[str, Any]] = []
    for i, name in enumerate(names):
        name = clean_text(name)
        if not name:
            continue
        topics.append(
            {
                'name': name,
                'category': clean_text(categories[i] if i < len(categories) else None),
                'weight': float(weights[i]) if i < len(weights) and weights[i] is not None else None,
                'position': i,
            }
        )
    return {'video_id': video_id, 'topics': topics}


SYNC_TIKTOK_COMMENT_TOPICS_CYPHER = """
UNWIND $rows AS row
OPTIONAL MATCH (v:TikTokVideo {video_id: row.video_id})
WITH row, v
WHERE v IS NOT NULL
OPTIONAL MATCH (v)-[r:HAS_COMMENT_TOPIC]->()
DELETE r
WITH row, v
UNWIND row.topics AS topic
WITH row, v, topic
WHERE topic.name IS NOT NULL
MERGE (ct:TikTokCommentTopic {video_id: row.video_id, name: topic.name})
SET ct.category = topic.category,
    ct.platform = $tiktok_platform
MERGE (v)-[rel:HAS_COMMENT_TOPIC]->(ct)
SET rel.weight = topic.weight,
    rel.position = topic.position,
    rel.platform = $tiktok_platform
"""


def ensure_tiktok_topic_constraint(driver) -> None:
    stmt = (
        'CREATE CONSTRAINT tiktok_comment_topic_video_name IF NOT EXISTS '
        'FOR (t:TikTokCommentTopic) REQUIRE (t.video_id, t.name) IS UNIQUE'
    )
    with driver.session(database=NEO4J_DATABASE) as s:
        s.run(stmt)
    print('Ensured constraint on (TikTokCommentTopic.video_id, TikTokCommentTopic.name)')


def ensure_tiktok_topic_vector_index(driver, dim: int) -> None:
    stmt = f"""CREATE VECTOR INDEX tiktok_comment_topic_embedding_index IF NOT EXISTS
    FOR (t:TikTokCommentTopic) ON (t.embedding)
    OPTIONS {{ indexConfig: {{ `vector.dimensions`: {dim}, `vector.similarity_function`: 'cosine' }} }}"""
    with driver.session(database=NEO4J_DATABASE) as s:
        s.run(stmt)
    print('Ensured tiktok_comment_topic_embedding_index')


def sync_tiktok_comment_topics_from_mongo(
    driver,
    collection,
    batch_size: int = TOPIC_UPSERT_BATCH,
) -> tuple[int, int]:
    """Returns (mongo_docs_in_batches, topic_slot_rows_synced) — count of topic entries processed (≤10×videos when capped)."""
    cur = collection.find({}, TIKTOK_TOPICS_MONGO_PROJECTION)
    if TOPIC_SYNC_LIMIT > 0:
        cur = cur.limit(TOPIC_SYNC_LIMIT)
    total_docs = 0
    total_links = 0
    for batch in batched(cur, batch_size):
        rows = []
        for doc in batch:
            row = topic_row_from_mongo_doc(doc)
            if row:
                rows.append(row)
        if not rows:
            continue
        total_docs += len(rows)
        batch_topic_slots = sum(len(r['topics']) for r in rows)
        with driver.session(database=NEO4J_DATABASE) as s:
            s.run(
                SYNC_TIKTOK_COMMENT_TOPICS_CYPHER,
                rows=rows,
                tiktok_platform=TIKTOK_PLATFORM,
            )
        total_links += batch_topic_slots
        print(f'  topic sync: mongo_rows={total_docs}  topic_slots_synced={total_links}')
    return total_docs, total_links


TIKTOK_TOPIC_FETCH_CYPHER = """
MATCH (t:TikTokCommentTopic)
WHERE coalesce(t.platform, 'tiktok') = $platform
  AND t.embedding IS NULL AND t.name IS NOT NULL
RETURN elementId(t) AS eid, t.name AS name
LIMIT $limit
"""

TIKTOK_TOPIC_WRITE_CYPHER = """
UNWIND $rows AS row
MATCH (t) WHERE elementId(t) = row.eid
CALL db.create.setNodeVectorProperty(t, 'embedding', row.embedding)
RETURN count(*) AS written
"""


def _embed_texts_for_tiktok_topics(texts: List[str], max_retries: int = 5) -> List[List[float]]:
    """Same logic as `embed_texts` in the summary section — uses embed_client from setup so topic embedding works without running summary cells."""
    out: List[List[float]] = []
    for chunk in batched(texts, EMBED_BATCH):
        attempt = 0
        while True:
            try:
                resp = embed_client.embeddings.create(model=EMBED_MODEL_NAME, input=chunk)
                out.extend([d.embedding for d in resp.data])
                break
            except Exception as e:
                attempt += 1
                if attempt > max_retries:
                    raise
                wait = min(2 ** attempt, 30)
                print(f'  embed retry {attempt} after {wait}s: {e}')
                time.sleep(wait)
    return out


def _write_tiktok_topic_embedding_chunk(driver, rows_chunk):
    """Write topic vectors in small Neo4j transactions; retry on transient/store errors (Aura)."""
    from neo4j.exceptions import Neo4jError

    attempt = 0
    while True:
        try:
            with driver.session(database=NEO4J_DATABASE) as s:
                rec = s.run(TIKTOK_TOPIC_WRITE_CYPHER, rows=rows_chunk).single()
                return int(rec['written']) if rec else 0
        except Neo4jError as e:
            attempt += 1
            if attempt > 8:
                raise
            wait = min(2 ** attempt, 120)
            print(f'  neo4j vector write retry {attempt} after {wait}s: {e}')
            time.sleep(wait)

def embed_tiktok_topic_nodes(driver, page_size: int = PAGE_SIZE) -> int:
    total_written = 0
    start = time.time()
    while True:
        with driver.session(database=NEO4J_DATABASE) as s:
            pending = list(
                s.run(
                    TIKTOK_TOPIC_FETCH_CYPHER,
                    limit=page_size,
                    platform=TIKTOK_PLATFORM,
                )
            )
        if not pending:
            break
        texts = [r['name'] for r in pending]
        embeddings = _embed_texts_for_tiktok_topics(texts)
        rows = [
            {'eid': pending[i]['eid'], 'embedding': embeddings[i]}
            for i in range(len(pending))
        ]
        for chunk in batched(rows, TOPIC_EMBED_WRITE_BATCH):
            written = _write_tiktok_topic_embedding_chunk(driver, chunk)
            total_written += written
        print(f'  TikTokCommentTopic embeddings written={total_written}  elapsed={time.time() - start:.1f}s')
    return total_written



In [ ]:
# Sync topics from Mongo, then embed TikTokCommentTopic.name
# Requires: setup cell (`driver`, `mongo_coll`, `embed_client`, `EMBED_DIM`) and — if you want summary/content vectors — those sections above.

ensure_tiktok_topic_constraint(driver)
mongo_docs, topic_links = sync_tiktok_comment_topics_from_mongo(driver, mongo_coll)
print('Topic sync done. mongo_docs (batched rows):', mongo_docs, '  topic_slots:', topic_links)

ensure_tiktok_topic_vector_index(driver, EMBED_DIM)
topic_embed_written = embed_tiktok_topic_nodes(driver)
print('Topic embedding rows written:', topic_embed_written)

with driver.session(database=NEO4J_DATABASE) as s:
    stats = {
        'TikTokCommentTopic': s.run('MATCH (t:TikTokCommentTopic) RETURN count(t) AS c').single()['c'],
        'HAS_COMMENT_TOPIC': s.run('MATCH ()-[r:HAS_COMMENT_TOPIC]->(:TikTokCommentTopic) RETURN count(r) AS c').single()['c'],
        'topics_with_embedding': s.run(
            "MATCH (t:TikTokCommentTopic) WHERE t.embedding IS NOT NULL RETURN count(t) AS c"
        ).single()['c'],
    }
print(stats)


## Test queries (TikTok)

These are the retrieval primitives the AI chat app will use. Each takes a theme (e.g. `"vaping"`, `"imagery of weapons or violence"`, `"gambling"`), embeds it, then hits one of the Neo4j vector indexes. All accept optional `k` (vector top-k recall) and `min_score` (cosine similarity floor).

- **`top_videos_by_topic_engagement(theme)`** – topic-matched videos ranked by *topic strength × log1p(comment_count)*. Best for *"most popular videos where the audience discusses X the most"*.
- **`top_creators_by_topic_engagement(theme)`** – same signal aggregated per creator. Best for *"most popular creators where audiences discuss X the most"*.
- **`top_creators(theme)`** – vector search over `Topic.embedding`, aggregate weighted contribution per `TikTokUser / username`. Best for *"which influencers' audiences talk about X"* (ranked top-N).
- **`count_creators_by_topic(theme, min_score=..., k=...)`** – count distinct creators with at least one comment-topic match at or above `min_score`. Best for *"how many creators talk about X"* (comment-discussion signal).
- **`count_creators_by_content(theme, min_score=..., k=...)`**
- **`count_videos_by_topic(theme)`** – count videos with comment-topic matches for X.
- **`count_videos_by_comment_summary(theme)`** – count videos whose comment-section summary matches X.
- **`count_videos_by_content(theme)`** – count videos whose on-video content matches X.
 – same count over `video_content_embedding`. Best for *"how many creators make content about X"*.
- **`example_videos(theme)`** – vector search over `Topic.embedding`, return videos whose comment-topics match. Best for *"show me examples of X content"* when you want comment-driven evidence.
- **`comment_discussions(theme)`** – vector search over `comment_summary_embedding`. Best for *"whose comment section discusses X"*.
- **`content_videos(theme)`** – vector search over `video_content_embedding` (title + description + thumbnail description + thumbnail keywords + tags). Best for **"which videos show / are about X"**, including visual imagery questions.
- **`content_top_creators(theme)`** – aggregates `content_videos` hits by creator.
- **`unified_search(theme)`** – runs all three vector indexes and fuses results with Reciprocal Rank Fusion (RRF). Use this when you don't know which signal the user's question maps to.


In [34]:
def embed_query(text: str) -> List[float]:
    """Embed a single user query string with the same model used for graph indexes."""
    return embed_texts([text])[0]

# Returns channels ranked by the sum of weighted topic-match contributions.
TOP_CREATORS_CYPHER = '''
CALL db.index.vector.queryNodes('tiktok_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'tiktok') = $platform
  AND score >= $min_score
MATCH (v:TikTokVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'tiktok') = $platform
OPTIONAL MATCH (u:TikTokUser)-[hv:HAS_VIDEO]->(v)
WHERE coalesce(hv.platform, "TikTok") = "TikTok"
WITH
  coalesce(u.username, v.username, 'unknown') AS creator,
  u.username AS username,
  t, v, rel, score
WITH
  creator,
  username,
  sum(score * coalesce(rel.weight, 1.0)) AS relevance,
  count(DISTINCT v) AS video_count,
  collect(DISTINCT t.name)[0..5] AS sample_topics,
  collect(DISTINCT {video_id: v.video_id, title: v.video_title, url: v.video_url})[0..8] AS sample_videos
RETURN creator, username, relevance, video_count, sample_topics, sample_videos
ORDER BY relevance DESC
LIMIT $top_n
'''
#the same as top_creators but for videos
EXAMPLE_VIDEOS_CYPHER = '''
CALL db.index.vector.queryNodes('tiktok_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'tiktok') = $platform
  AND score >= $min_score
MATCH (v:TikTokVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'tiktok') = $platform
WITH v, t, rel, score
ORDER BY score * coalesce(rel.weight, 1.0) DESC
WITH v,
     collect({topic: t.name, weight: rel.weight, score: score})[0..3] AS matches,
     max(score * coalesce(rel.weight, 1.0)) AS best
RETURN v.video_id AS video_id,
       v.video_title AS title,
       coalesce(v.username, '') AS creator,
       v.video_url AS url,
       v.view_count AS views,
       best AS relevance,
       matches
ORDER BY best DESC
LIMIT $top_n
'''
#returns top n videos with the highest similarity score from comment_summary_embedding
COMMENT_DISCUSSIONS_CYPHER = '''
CALL db.index.vector.queryNodes('tiktok_video_summary_embedding_index', $k, $q) YIELD node AS v, score
WHERE score >= $min_score
RETURN v.video_id AS video_id,
       v.video_title AS title,
       coalesce(v.username, '') AS creator,
       coalesce(v.username, '') AS username,
       v.video_url AS url,
       score,
       v.comment_summary_description AS comment_summary_description
ORDER BY score DESC
LIMIT $top_n
'''
#returns top n videos with the highest similarity score from video_content_embedding
CONTENT_VIDEOS_CYPHER = '''
CALL db.index.vector.queryNodes('tiktok_video_content_embedding_index', $k, $q) YIELD node AS v, score
WHERE score >= $min_score
RETURN v.video_id AS video_id,
       v.video_title AS title,
       coalesce(v.username, '') AS creator,
       coalesce(v.username, '') AS username,
       v.video_url AS url,
       v.view_count AS views,
       v.thumbnail_description AS thumbnail_description,
       v.video_description AS video_description,
       v.thumbnail_keywords AS thumbnail_keywords,
       v.hashtag_names AS hashtag_names,
       score
ORDER BY score DESC
LIMIT $top_n
'''
# Returns channels ranked by the  sum of video_content_embedding similarity scores.
CONTENT_TOP_CREATORS_CYPHER = '''
CALL db.index.vector.queryNodes('tiktok_video_content_embedding_index', $k, $q) YIELD node AS v, score
WHERE score >= $min_score
OPTIONAL MATCH (u:TikTokUser)-[hv:HAS_VIDEO]->(v)
WHERE coalesce(hv.platform, "TikTok") = "TikTok"
WITH coalesce(u.username, v.username, 'unknown') AS creator,
     u.username AS username,
     v, score
WITH creator, username,
     sum(score) AS relevance,
     count(DISTINCT v) AS video_count,
     collect({video_id: v.video_id, title: v.video_title, url: v.video_url, score: score})[0..5] AS sample_videos
RETURN creator, username, relevance, video_count, sample_videos
ORDER BY relevance DESC
LIMIT $top_n
'''

# Count distinct creators with topic matches at or above min_score (bounded by vector top-k).
COUNT_CREATORS_BY_TOPIC_CYPHER = '''
CALL db.index.vector.queryNodes('tiktok_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'tiktok') = $platform
  AND score >= $min_score
MATCH (v:TikTokVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'tiktok') = $platform
OPTIONAL MATCH (u:TikTokUser)-[hv:HAS_VIDEO]->(v)
WHERE coalesce(hv.platform, "TikTok") = "TikTok"
WITH
  coalesce(u.username, v.username, 'unknown') AS creator,
  u.username AS username,
  v,
  score * coalesce(rel.weight, 1.0) AS weighted_score
WITH creator, username,
     count(DISTINCT v) AS video_count,
     max(weighted_score) AS best_weighted_score
RETURN count(*) AS creator_count,
       sum(video_count) AS video_link_rows,
       collect({creator: creator, username: username, video_count: video_count, best_weighted_score: best_weighted_score})[0..20] AS sample_creators
'''

# Count distinct creators with video-content matches at or above min_score.
COUNT_CREATORS_BY_CONTENT_CYPHER = '''
CALL db.index.vector.queryNodes('tiktok_video_content_embedding_index', $k, $q) YIELD node AS v, score
WHERE score >= $min_score
OPTIONAL MATCH (u:TikTokUser)-[hv:HAS_VIDEO]->(v)
WHERE coalesce(hv.platform, "TikTok") = "TikTok"
WITH
  coalesce(u.username, v.username, 'unknown') AS creator,
  u.username AS username,
  v,
  score
WITH creator, username,
     count(DISTINCT v) AS video_count,
     max(score) AS best_score
RETURN count(*) AS creator_count,
       sum(video_count) AS video_link_rows,
       collect({creator: creator, username: username, video_count: video_count, best_score: best_score})[0..20] AS sample_creators
'''

# Count distinct videos with comment-topic matches at or above min_score.
COUNT_VIDEOS_BY_TOPIC_CYPHER = '''
CALL db.index.vector.queryNodes('tiktok_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'tiktok') = $platform
  AND score >= $min_score
MATCH (v:TikTokVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'tiktok') = $platform
WITH v, score * coalesce(rel.weight, 1.0) AS weighted_score
WITH v, max(weighted_score) AS best_weighted_score
RETURN count(v) AS video_count,
       collect({
         video_id: v.video_id,
         title: v.video_title,
         creator: coalesce(v.username, ''),
         url: v.video_url,
         best_weighted_score: best_weighted_score
       })[0..20] AS sample_videos
'''

# Count distinct videos whose comment-summary embedding matches at or above min_score.
COUNT_VIDEOS_BY_COMMENT_SUMMARY_CYPHER = '''
CALL db.index.vector.queryNodes('tiktok_video_summary_embedding_index', $k, $q) YIELD node AS v, score
WHERE score >= $min_score
WITH v, max(score) AS best_score
RETURN count(v) AS video_count,
       collect({
         video_id: v.video_id,
         title: v.video_title,
         creator: coalesce(v.username, ''),
         url: v.video_url,
         score: best_score
       })[0..20] AS sample_videos
'''

# Count distinct videos with video-content matches at or above min_score.
COUNT_VIDEOS_BY_CONTENT_CYPHER = '''
CALL db.index.vector.queryNodes('tiktok_video_content_embedding_index', $k, $q) YIELD node AS v, score
WHERE score >= $min_score
WITH v, max(score) AS best_score
RETURN count(v) AS video_count,
       collect({
         video_id: v.video_id,
         title: v.video_title,
         creator: coalesce(v.username, ''),
         url: v.video_url,
         score: best_score
       })[0..20] AS sample_videos
'''
# Rank videos by topic match strength × log1p(comment_count).
TOP_VIDEOS_BY_TOPIC_ENGAGEMENT_CYPHER = '''
CALL db.index.vector.queryNodes('tiktok_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'tiktok') = $platform
  AND score >= $min_score
MATCH (v:TikTokVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'tiktok') = $platform
WITH v, t, rel, score,
  score * coalesce(rel.weight, 1.0) AS topic_strength,
  log(1.0 + toFloat(coalesce(v.comment_count, 0))) AS log_comments
WITH v,
  max(topic_strength) AS best_topic_strength,
  max(log_comments) AS log_comments,
  collect(DISTINCT {topic: t.name, weight: rel.weight, score: score})[0..3] AS matches
WITH v, best_topic_strength, log_comments, matches,
  best_topic_strength * log_comments AS engagement_score
RETURN v.video_id AS video_id,
       v.video_title AS title,
       coalesce(v.username, '') AS creator,
       coalesce(v.username, '') AS username,
       v.video_url AS url,
       toInteger(coalesce(v.comment_count, 0)) AS comment_count,
       round(best_topic_strength, 5) AS best_topic_strength,
       round(log_comments, 5) AS log_comments,
       round(engagement_score, 5) AS engagement_score,
       matches
ORDER BY engagement_score DESC
LIMIT $top_n
'''

# Rank creators by summed per-video engagement (topic strength × log comments).
TOP_CREATORS_BY_TOPIC_ENGAGEMENT_CYPHER = '''
CALL db.index.vector.queryNodes('tiktok_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'tiktok') = $platform
  AND score >= $min_score
MATCH (v:TikTokVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'tiktok') = $platform
OPTIONAL MATCH (u:TikTokUser)-[hv:HAS_VIDEO]->(v)
WHERE coalesce(hv.platform, "TikTok") = "TikTok"
WITH
  coalesce(u.username, v.username, 'unknown') AS creator,
  u.username AS username,
  v, t, rel, score,
  score * coalesce(rel.weight, 1.0) AS topic_strength,
  log(1.0 + toFloat(coalesce(v.comment_count, 0))) AS log_comments
WITH creator, username, v,
  max(topic_strength) AS video_topic_strength,
  max(log_comments) AS log_comments,
  collect(DISTINCT t.name)[0..5] AS video_topics
WITH creator, username, v, video_topics,
  video_topic_strength * log_comments AS video_engagement,
  toInteger(coalesce(v.comment_count, 0)) AS comment_count
WITH creator, username,
  sum(video_engagement) AS engagement_score,
  count(DISTINCT v) AS video_count,
  sum(comment_count) AS total_comment_count,
  collect(DISTINCT video_topics)[0..5] AS sample_topics,
  collect({
    video_id: v.video_id,
    title: v.video_title,
    url: v.video_url,
    comment_count: comment_count,
    engagement_score: round(video_engagement, 5)
  })[0..8] AS sample_videos
RETURN creator,
       username,
       round(engagement_score, 5) AS engagement_score,
       video_count,
       total_comment_count,
       sample_topics,
       sample_videos
ORDER BY engagement_score DESC
LIMIT $top_n
'''



def top_creators(theme: str, top_n: int = 10, k: int = 2000, min_score: float = 0.0):
    """Rank creators whose audiences discuss `theme` (comment-topic signal).

    Index: `tiktok_comment_topic_embedding_index` on `TikTokCommentTopic`.
    Ranks by: sum(vector_score × rel.weight) per channel.
    Best for: *"which creators have audiences discussing X?"* (quality / relevance, top-N).
    Returns: creator, username, relevance, video_count, sample_topics, sample_videos.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(
            TOP_CREATORS_CYPHER,
            q=q,
            k=max(k, top_n),
            top_n=top_n,
            min_score=min_score,
            platform=TIKTOK_PLATFORM,
        )]


def count_creators_by_topic(
    theme: str,
    min_score: float = 0.0,
    k: int = 2000,
) -> Dict[str, Any]:
    """Count distinct creators with comment-topic matches for `theme` at or above `min_score`.

    Index: `tiktok_comment_topic_embedding_index` (same joins as `top_creators`).
    Best for: *"how many creators talk about X in comments?"* (scale, not a ranked list).
    Returns: creator_count, video_link_rows, sample_creators; bounded by vector top-k.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        rec = s.run(
            COUNT_CREATORS_BY_TOPIC_CYPHER,
            q=q,
            k=k,
            min_score=min_score,
            platform=TIKTOK_PLATFORM,
        ).single()
    return {
        'theme': theme,
        'signal': 'comment_topics',
        'min_score': min_score,
        'k': k,
        'creator_count': int(rec['creator_count']) if rec else 0,
        'video_link_rows': int(rec['video_link_rows']) if rec else 0,
        'sample_creators': list(rec['sample_creators'] or []),
    }


def count_creators_by_content(
    theme: str,
    min_score: float = 0.0,
    k: int = 2000,
) -> Dict[str, Any]:
    """Count distinct creators with video-content matches for `theme` at or above `min_score`.

    Index: `tiktok_video_content_embedding_index` on `TikTokVideo` (title, description, thumbnail, tags).
    Best for: *"how many creators make content about X?"* (not comment-audience signal).
    Returns: creator_count, video_link_rows, sample_creators; bounded by vector top-k.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        rec = s.run(
            COUNT_CREATORS_BY_CONTENT_CYPHER,
            q=q,
            k=k,
            min_score=min_score,
        ).single()
    return {
        'theme': theme,
        'signal': 'video_content',
        'min_score': min_score,
        'k': k,
        'creator_count': int(rec['creator_count']) if rec else 0,
        'video_link_rows': int(rec['video_link_rows']) if rec else 0,
        'sample_creators': list(rec['sample_creators'] or []),
    }


def count_videos_by_topic(
    theme: str,
    min_score: float = 0.0,
    k: int = 2000,
) -> Dict[str, Any]:
    """Count distinct videos with comment-topic matches for `theme` at or above `min_score`.

    Index: `tiktok_comment_topic_embedding_index` (same path as `example_videos`).
    Best for: *"how many videos discuss X in comment topics?"*
    Returns: video_count, sample_videos; bounded by vector top-k.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        rec = s.run(
            COUNT_VIDEOS_BY_TOPIC_CYPHER,
            q=q,
            k=k,
            min_score=min_score,
            platform=TIKTOK_PLATFORM,
        ).single()
    return {
        'theme': theme,
        'signal': 'comment_topics',
        'min_score': min_score,
        'k': k,
        'video_count': int(rec['video_count']) if rec else 0,
        'sample_videos': list(rec['sample_videos'] or []),
    }


def count_videos_by_comment_summary(
    theme: str,
    min_score: float = 0.0,
    k: int = 2000,
) -> Dict[str, Any]:
    """Count distinct videos whose comment-section summary matches `theme`.

    Index: `tiktok_video_summary_embedding_index` (same path as `comment_discussions`).
    Best for: *"how many videos have comment sections that discuss X?"*
    Returns: video_count, sample_videos; bounded by vector top-k.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        rec = s.run(
            COUNT_VIDEOS_BY_COMMENT_SUMMARY_CYPHER,
            q=q,
            k=k,
            min_score=min_score,
        ).single()
    return {
        'theme': theme,
        'signal': 'comment_summary',
        'min_score': min_score,
        'k': k,
        'video_count': int(rec['video_count']) if rec else 0,
        'sample_videos': list(rec['sample_videos'] or []),
    }


def count_videos_by_content(
    theme: str,
    min_score: float = 0.0,
    k: int = 2000,
) -> Dict[str, Any]:
    """Count distinct videos whose on-video content matches `theme`.

    Index: `tiktok_video_content_embedding_index` (same path as `content_videos`).
    Best for: *"how many videos are about / show X?"* (not comment signal).
    Returns: video_count, sample_videos; bounded by vector top-k.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        rec = s.run(
            COUNT_VIDEOS_BY_CONTENT_CYPHER,
            q=q,
            k=k,
            min_score=min_score,
        ).single()
    return {
        'theme': theme,
        'signal': 'video_content',
        'min_score': min_score,
        'k': k,
        'video_count': int(rec['video_count']) if rec else 0,
        'sample_videos': list(rec['sample_videos'] or []),
    }


def top_videos_by_topic_engagement(theme: str, top_n: int = 10, k: int = 2000, min_score: float = 0.0):
    """Rank videos where audiences discuss `theme` the most at comment volume (engagement).

    Index: `tiktok_comment_topic_embedding_index` + `HAS_COMMENT_TOPIC` weights.
    Ranks by: max(score × weight) per video × log1p(comment_count).
    Best for: *"most popular videos where people discuss X in comments"*.
    Returns: video_id, title, creator, comment_count, engagement_score, matches, …
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(
            TOP_VIDEOS_BY_TOPIC_ENGAGEMENT_CYPHER,
            q=q,
            k=max(k, top_n),
            top_n=top_n,
            min_score=min_score,
            platform=TIKTOK_PLATFORM,
        )]


def top_creators_by_topic_engagement(theme: str, top_n: int = 10, k: int = 2000, min_score: float = 0.0):
    """Rank creators by summed discussion engagement for `theme` (topic match × comment volume).

    Index: same as `top_videos_by_topic_engagement`, aggregated per channel.
    Ranks by: sum over videos of max(score × weight) × log1p(comment_count).
    Best for: *"which popular creators have the biggest audience discussions about X?"*.
    Returns: engagement_score, video_count, total_comment_count, sample_topics, sample_videos.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(
            TOP_CREATORS_BY_TOPIC_ENGAGEMENT_CYPHER,
            q=q,
            k=max(k, top_n),
            top_n=top_n,
            min_score=min_score,
            platform=TIKTOK_PLATFORM,
        )]



def example_videos(theme: str, top_n: int = 10, k: int = 2000, min_score: float = 0.0):
    """Return example videos with the strongest comment-topic matches for `theme`.

    Index: `tiktok_comment_topic_embedding_index` on `TikTokCommentTopic`.
    Ranks by: best score × rel.weight per video; up to 3 matching topic names per video.
    Best for: *"show me videos where commenters discuss X"* (evidence / examples).
    Returns: video_id, title, creator, url, views, relevance, matches.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(
            EXAMPLE_VIDEOS_CYPHER,
            q=q,
            k=max(k, top_n),
            top_n=top_n,
            min_score=min_score,
            platform=TIKTOK_PLATFORM,
        )]

def comment_discussions(theme: str, top_n: int = 10, k: int = 2000, min_score: float = 0.0):
    """Find videos whose comment-section summary is semantically similar to `theme`.

    Index: `tiktok_video_summary_embedding_index` on `comment_summary_embedding`.
    Ranks by: raw vector similarity (no per-topic weights).
    Best for: *"whose comment section discusses X?"* / narrative comment themes.
    Returns: video_id, title, creator, url, score, comment_summary_description.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(
            COMMENT_DISCUSSIONS_CYPHER,
            q=q,
            k=max(k, top_n),
            top_n=top_n,
            min_score=min_score,
        )]

def content_videos(theme: str, top_n: int = 10, k: int = 2000, min_score: float = 0.0):
    """Find videos whose on-video content matches `theme` (not comment topics).

    Index: `tiktok_video_content_embedding_index` (title, description, thumbnail text/keywords, tags).
    Ranks by: vector similarity on video content embedding.
    Best for: *"which videos are about / show X?"* including visual or title/description cues.
    Returns: video_id, title, creator, url, views, thumbnail fields, tags, score.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(
            CONTENT_VIDEOS_CYPHER,
            q=q,
            k=max(k, top_n),
            top_n=top_n,
            min_score=min_score,
        )]

def content_top_creators(theme: str, top_n: int = 10, k: int = 2000, min_score: float = 0.0):
    """Rank creators by how much their published videos match `theme` (content signal).

    Index: `tiktok_video_content_embedding_index`; aggregates content hits per channel.
    Ranks by: sum(content vector scores) across matching videos.
    Best for: *"which creators make content about X?"* (not audience comment topics).
    Returns: creator, username, relevance, video_count, sample_videos.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(
            CONTENT_TOP_CREATORS_CYPHER,
            q=q,
            k=max(k, top_n),
            top_n=top_n,
            min_score=min_score,
            platform=TIKTOK_PLATFORM,
        )]


# --- Unified fused search ---------------------------------------------------
# Runs all three video-level indexes and fuses rankings with Reciprocal Rank
# Fusion (RRF). RRF is robust against differences in score scales between
# indexes and is the standard way to combine heterogeneous retrievers.

UNIFIED_CONTENT_Q = '''
CALL db.index.vector.queryNodes('tiktok_video_content_embedding_index', $k, $q) YIELD node AS v, score
WHERE score >= $min_score
RETURN v.video_id AS video_id, score
'''

UNIFIED_SUMMARY_Q = '''
CALL db.index.vector.queryNodes('tiktok_video_summary_embedding_index', $k, $q) YIELD node AS v, score
WHERE score >= $min_score
RETURN v.video_id AS video_id, score
'''

UNIFIED_TOPIC_Q = '''
CALL db.index.vector.queryNodes('tiktok_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'tiktok') = $platform
  AND score >= $min_score
MATCH (v:TikTokVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'tiktok') = $platform
WITH v.video_id AS video_id, max(score * coalesce(rel.weight, 1.0)) AS score
RETURN video_id, score
'''

UNIFIED_HYDRATE = '''
UNWIND $video_ids AS vid
MATCH (v:TikTokVideo {video_id: vid})
OPTIONAL MATCH (u:TikTokUser)-[hv:HAS_VIDEO]->(v)
WHERE coalesce(hv.platform, "TikTok") = "TikTok"
RETURN v.video_id AS video_id,
       v.video_title AS title,
       coalesce(u.username, v.username) AS creator,
       u.username AS username,
       v.video_url AS url,
       v.view_count AS views,
       v.thumbnail_description AS thumbnail_description,
       v.video_description AS video_description,
       v.comment_summary_description AS comment_summary_description
'''


def _rrf_rank(rows, id_key: str = 'video_id', k_const: int = 60) -> Dict[str, float]:
    """Convert an ordered hit list into Reciprocal Rank Fusion scores: 1 / (k_const + rank + 1)."""
    scores: Dict[str, float] = {}
    for rank, row in enumerate(rows):
        scores[row[id_key]] = 1.0 / (k_const + rank + 1)
    return scores

def unified_search(
    theme: str,
    top_n: int = 10,
    k: int = 2000,
    min_score: float = 0.0,
    include_creator_rollup: bool = True,
):
    """Fuse video hits from content, comment-summary, and comment-topic indexes (RRF).

    Indexes: `tiktok_video_content_embedding_index`, `tiktok_video_summary_embedding_index`,
    `tiktok_comment_topic_embedding_index`.
    Ranks by: sum of reciprocal-rank scores per video across lists (robust to score scale).
    Best for: ambiguous queries — use when you are unsure which signal fits the question.
    Returns: {videos: [...], creators: [...]} with matched_signals and fused_score per video.
    """
    q = embed_query(theme)
    k_vec = max(k, top_n)
    with driver.session(database=NEO4J_DATABASE) as s:
        content_rows = [dict(r) for r in s.run(UNIFIED_CONTENT_Q, q=q, k=k_vec, min_score=min_score)]
        summary_rows = [dict(r) for r in s.run(UNIFIED_SUMMARY_Q, q=q, k=k_vec, min_score=min_score)]
        topic_rows = [dict(r) for r in s.run(
            UNIFIED_TOPIC_Q, q=q, k=k_vec, min_score=min_score, platform=TIKTOK_PLATFORM
        )]

        content_rank = _rrf_rank(content_rows)
        summary_rank = _rrf_rank(summary_rows)
        topic_rank   = _rrf_rank(topic_rows)

        fused: Dict[str, float] = {}
        signals: Dict[str, List[str]] = {}
        for ranks, label in [(content_rank, 'content'), (summary_rank, 'comments'), (topic_rank, 'topic')]:
            for vid, s_val in ranks.items():
                fused[vid] = fused.get(vid, 0.0) + s_val
                signals.setdefault(vid, []).append(label)

        top_ids = sorted(fused.keys(), key=lambda x: fused[x], reverse=True)[:top_n]
        if not top_ids:
            return {'videos': [], 'creators': []}

        hydrated = {r['video_id']: dict(r) for r in s.run(UNIFIED_HYDRATE, video_ids=top_ids, platform=TIKTOK_PLATFORM)}
        videos = []
        for vid in top_ids:
            if vid not in hydrated:
                continue
            row = hydrated[vid]
            row['fused_score'] = round(fused[vid], 5)
            row['matched_signals'] = signals[vid]
            videos.append(row)

        creators = []
        if include_creator_rollup:
            rollup: Dict[str, Dict[str, Any]] = {}
            for row in videos:
                cid = row.get('username') or row.get('creator') or 'unknown'
                agg = rollup.setdefault(cid, {
                    'username': cid,
                    'creator': row['creator'],
                    'relevance': 0.0,
                    'video_count': 0,
                    'videos': [],
                })
                agg['relevance'] += row['fused_score']
                agg['video_count'] += 1
                agg['videos'].append({
                    'video_id': row['video_id'],
                    'title': row['title'],
                    'url': row.get('url'),
                })
            creators = sorted(rollup.values(), key=lambda x: x['relevance'], reverse=True)

    return {'videos': videos, 'creators': creators}


### Run test queries

Set `THEME`, `TOP_N`, `MIN_SCORE`, and `K_COUNT` in the next cell, then run each retriever cell below. All retrievers use `MIN_SCORE` and `K_COUNT` from the config cell.


In [18]:
from pprint import pprint

THEME = 'gaming'
TOP_N = 10
MIN_SCORE = 0.75   # raise after similarity analysis (e.g. 0.75 for topics)
K_COUNT = 20000    # vector top-k for all retrievers

print(f'THEME: {THEME}')
print(f'TOP_N: {TOP_N}')
print(f'MIN_SCORE: {MIN_SCORE}')
print(f'K_COUNT: {K_COUNT}')


THEME: gaming
TOP_N: 10
MIN_SCORE: 0.75
K_COUNT: 20000


In [20]:
print(f'--- top_creators: comment-topic audience for "{THEME}" ---')
for row in top_creators(THEME, top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE):
    pprint(row)


--- top_creators: comment-topic audience for "gaming" ---
{'creator': 'breckiehill',
 'relevance': 1.4513635396957403,
 'sample_topics': ['Gaming and Gamers',
                   'Gaming (Fortnite)',
                   'Gaming/Fortnite',
                   'Gaming References',
                   'Fortnite and Gaming'],
 'sample_videos': [{'title': '',
                    'url': 'https://www.tiktok.com/@breckiehill/video/7288202662183521578',
                    'video_id': 7288202662183521578},
                   {'title': '',
                    'url': 'https://www.tiktok.com/@breckiehill/video/7279509248399133998',
                    'video_id': 7279509248399133998},
                   {'title': '',
                    'url': 'https://www.tiktok.com/@breckiehill/video/7310212500325453102',
                    'video_id': 7310212500325453102},
                   {'title': 'oh of course',
                    'url': 'https://www.tiktok.com/@breckiehill/video/7298154911169203502',
      

In [21]:
print(f'--- top_videos_by_topic_engagement: popular videos discussing "{THEME}" ---')
for row in top_videos_by_topic_engagement(THEME, top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE):
    pprint(row)


--- top_videos_by_topic_engagement: popular videos discussing "gaming" ---
{'best_topic_strength': 0.36886,
 'comment_count': 3682,
 'creator': 'redbullracing',
 'engagement_score': 3.02888,
 'log_comments': 8.21148,
 'matches': [{'score': 0.7848072052001953,
              'topic': 'FIFA and Gaming',
              'weight': 0.47}],
 'title': 'This was too easy 😅 put your flag below ⬇️ #F1 #RedBullRacing '
          '#flagfilter #MaxVerstappen ',
 'url': 'https://www.tiktok.com/@redbullracing/video/7208983476031294726',
 'username': 'redbullracing',
 'video_id': 7208983476031294726}
{'best_topic_strength': 0.36627,
 'comment_count': 2172,
 'creator': 'sam_sulek',
 'engagement_score': 2.81437,
 'log_comments': 7.68386,
 'matches': [{'score': 0.7630629539489746,
              'topic': 'Clash Royale and Gaming',
              'weight': 0.48}],
 'title': 'I did my cardio today, did you? #hosstile #hosstilesupps '
          '#bodybuilding #physique #muscle #posing #lifting #lifter #lift #gym

In [22]:
print(f'--- top_creators_by_topic_engagement: popular creators discussing "{THEME}" ---')
for row in top_creators_by_topic_engagement(THEME, top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE):
    pprint(row)


--- top_creators_by_topic_engagement: popular creators discussing "gaming" ---
{'creator': 'breckiehill',
 'engagement_score': 8.70356,
 'sample_topics': [['Gaming and Gamers'],
                   ['Gaming (Fortnite)'],
                   ['Gaming/Fortnite'],
                   ['Gaming References'],
                   ['Fortnite and Gaming']],
 'sample_videos': [{'comment_count': 385,
                    'engagement_score': 0.71098,
                    'title': '',
                    'url': 'https://www.tiktok.com/@breckiehill/video/7288202662183521578',
                    'video_id': 7288202662183521578},
                   {'comment_count': 465,
                    'engagement_score': 0.52364,
                    'title': '',
                    'url': 'https://www.tiktok.com/@breckiehill/video/7279509248399133998',
                    'video_id': 7279509248399133998},
                   {'comment_count': 309,
                    'engagement_score': 0.2328,
                    'ti

In [23]:
print(f'--- count_creators_by_topic: how many creators discuss "{THEME}" ---')
pprint(count_creators_by_topic(THEME, min_score=MIN_SCORE, k=K_COUNT))


--- count_creators_by_topic: how many creators discuss "gaming" ---
{'creator_count': 32,
 'k': 20000,
 'min_score': 0.75,
 'sample_creators': [{'best_weighted_score': 0.32856349945068364,
                      'creator': 'pietervalley',
                      'username': 'pietervalley',
                      'video_count': 3},
                     {'best_weighted_score': 0.2054152488708496,
                      'creator': 'katie_sigmond',
                      'username': 'katie_sigmond',
                      'video_count': 6},
                     {'best_weighted_score': 0.09505667686462403,
                      'creator': 'sihamsanad_',
                      'username': 'sihamsanad_',
                      'video_count': 1},
                     {'best_weighted_score': 0.26610435485839845,
                      'creator': 'timonverbeeck',
                      'username': 'timonverbeeck',
                      'video_count': 2},
                     {'best_weighted_score': 0.23137

In [24]:
print(f'--- count_creators_by_content: how many creators publish about "{THEME}" ---')
pprint(count_creators_by_content(THEME, min_score=MIN_SCORE, k=K_COUNT))


--- count_creators_by_content: how many creators publish about "gaming" ---
{'creator_count': 0,
 'k': 20000,
 'min_score': 0.75,
 'sample_creators': [],
 'signal': 'video_content',
 'theme': 'gaming',
 'video_link_rows': 0}


In [25]:
print(f'--- count_videos_by_topic: videos with comment-topic matches for "{THEME}" ---')
pprint(count_videos_by_topic(THEME, min_score=MIN_SCORE, k=K_COUNT))


--- count_videos_by_topic: videos with comment-topic matches for "gaming" ---
{'k': 20000,
 'min_score': 0.75,
 'sample_videos': [{'best_weighted_score': 0.028519864082336425,
                    'creator': 'pietervalley',
                    'title': 'En dan hebben ze 849k likes 😭😂 ',
                    'url': 'https://www.tiktok.com/@pietervalley/video/7266823169414024480',
                    'video_id': 7266823169414024480},
                   {'best_weighted_score': 0.03802391052246094,
                    'creator': 'katie_sigmond',
                    'title': '',
                    'url': 'https://www.tiktok.com/@katie_sigmond/video/7275821209936301343',
                    'video_id': 7275821209936301343},
                   {'best_weighted_score': 0.09505667686462403,
                    'creator': 'sihamsanad_',
                    'title': '',
                    'url': 'https://www.tiktok.com/@sihamsanad_/video/7258252534509341978',
                    'video_id': 725825

In [26]:
print(f'--- count_videos_by_comment_summary: comment sections discussing "{THEME}" ---')
pprint(count_videos_by_comment_summary(THEME, min_score=MIN_SCORE, k=K_COUNT))


--- count_videos_by_comment_summary: comment sections discussing "gaming" ---
{'k': 20000,
 'min_score': 0.75,
 'sample_videos': [],
 'signal': 'comment_summary',
 'theme': 'gaming',
 'video_count': 0}


In [27]:
print(f'--- count_videos_by_content: videos about / showing "{THEME}" ---')
pprint(count_videos_by_content(THEME, min_score=MIN_SCORE, k=K_COUNT))


--- count_videos_by_content: videos about / showing "gaming" ---
{'k': 20000,
 'min_score': 0.75,
 'sample_videos': [],
 'signal': 'video_content',
 'theme': 'gaming',
 'video_count': 0}


In [28]:
print(f'--- example_videos: comment-topic evidence for "{THEME}" ---')
for row in example_videos(THEME, top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE):
    pprint(row)


--- example_videos: comment-topic evidence for "gaming" ---
{'creator': 'amustycow',
 'matches': [{'score': 0.7749075889587402,
              'topic': 'Gaming History Recall',
              'weight': 0.5}],
 'relevance': 0.3874537944793701,
 'title': 'the worlds biggest rocket league kahoot 👀 (part 2) #rocketleague ',
 'url': 'https://www.tiktok.com/@amustycow/video/7238000039086378282',
 'video_id': 7238000039086378282,
 'views': 106145}
{'creator': 'redbullracing',
 'matches': [{'score': 0.7848072052001953,
              'topic': 'FIFA and Gaming',
              'weight': 0.47}],
 'relevance': 0.3688593864440918,
 'title': 'This was too easy 😅 put your flag below ⬇️ #F1 #RedBullRacing '
          '#flagfilter #MaxVerstappen ',
 'url': 'https://www.tiktok.com/@redbullracing/video/7208983476031294726',
 'video_id': 7208983476031294726,
 'views': 4180641}
{'creator': 'sam_sulek',
 'matches': [{'score': 0.7630629539489746,
              'topic': 'Clash Royale and Gaming',
              '

In [29]:
print(f'--- comment_discussions: comment summaries matching "{THEME}" ---')
for row in comment_discussions(THEME, top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE):
    pprint(row)


--- comment_discussions: comment summaries matching "gaming" ---


In [30]:
print(f'--- content_videos: videos about / showing "{THEME}" ---')
for row in content_videos(THEME, top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE):
    pprint(row)


--- content_videos: videos about / showing "gaming" ---


In [31]:
print(f'--- content_top_creators: creators whose videos depict "{THEME}" ---')
for row in content_top_creators(THEME, top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE):
    pprint(row)


--- content_top_creators: creators whose videos depict "gaming" ---


In [32]:
print(f'--- unified_search: fused videos for "{THEME}" ---')
_fused = unified_search(THEME, top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE)
print('Top videos (signals that matched):')
for row in _fused['videos']:
    pprint(row)


--- unified_search: fused videos for "gaming" ---
Top videos (signals that matched):
{'comment_summary_description': 'The comments are lively and humorous, with '
                                "viewers joking about the video's appearance "
                                'on their For You Pages, tagging friends, and '
                                "referencing the creator's looks, height, and "
                                'use of trending filters. Many comments '
                                'encourage gaining more likes, highlight the '
                                'virality of the content, and make playful or '
                                'flirtatious remarks about the creator. The '
                                'overall sentiment is very positive and '
                                'light-hearted, with users sharing memes, '
                                'inside jokes, and expressing amusement or '
                                'admiration. Some viewers me

In [33]:
print(f'--- unified_search: fused creator rollup for "{THEME}" ---')
_fused = unified_search(THEME, top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE)
for row in _fused['creators']:
    pprint(row)


--- unified_search: fused creator rollup for "gaming" ---
{'creator': 'breckiehill',
 'relevance': 0.03009,
 'username': 'breckiehill',
 'video_count': 2,
 'videos': [{'title': '',
             'url': 'https://www.tiktok.com/@breckiehill/video/7288202662183521578',
             'video_id': 7288202662183521578},
            {'title': '',
             'url': 'https://www.tiktok.com/@breckiehill/video/7279509248399133998',
             'video_id': 7279509248399133998}]}
{'creator': 'timonverbeeck',
 'relevance': 0.02991,
 'username': 'timonverbeeck',
 'video_count': 2,
 'videos': [{'title': 'Hoe game jij? 😂🎮 Advertentie @bol | Mijn PlayStation 4 '
                      'is stuk, dus ik scoor een PS5 met korting op bol.com en '
                      'krijg er gratis EA FC 24 bij! 😱 #voorjou #allesopbol '
                      '#EASPORTSFC #FC24 ',
             'url': 'https://www.tiktok.com/@timonverbeeck/video/7288340095604690209',
             'video_id': 7288340095604690209},
          

## Similarity score analysis (production thresholds)

For each **query string**, run vector search against all **three** indexes (`video_content_embedding`, `comment_summary_embedding`, `topic_embedding`). For each index:

- print **query name** + index name
- show an **ASCII histogram** of cosine similarity `score` (20 bins on \[0, 1\], with spill counts outside that range)
- print up to **5 example rows** per score band: **0.90–1.00**, **0.80–0.89**, **0.70–0.79**, **0.60–0.69**, **0.50–0.59**

Tune `SIMILARITY_K_FRACTION` (default **0.1** = 10%), `SIMILARITY_K_MIN` / `SIMILARITY_K_MAX`, optional fixed override `SIMILARITY_ANALYSIS_K`, and `QUERIES_FOR_THRESHOLD_TUNING`, then run the next cell.


In [ ]:
import os
from pprint import pprint
from typing import Any, Dict, List, Optional, Tuple

# Align names with ensure_vector_indexes cell
TIKTOK_CONTENT_EMBED_IDX = 'tiktok_video_content_embedding_index'
TIKTOK_SUMMARY_EMBED_IDX = 'tiktok_video_summary_embedding_index'

SIMILARITY_K_FRACTION = float(os.getenv('SIMILARITY_K_FRACTION', '0.2'))
SIMILARITY_K_MIN = int(os.getenv('SIMILARITY_K_MIN', '50'))
SIMILARITY_K_MAX = int(os.getenv('SIMILARITY_K_MAX', '25000'))

SIMILARITY_K_OVERRIDE_RAW = os.getenv('SIMILARITY_ANALYSIS_K', '').strip()

_INDEXED_NODE_COUNTS_CYPHER: Dict[str, str] = {
    TIKTOK_CONTENT_EMBED_IDX: (
        'MATCH (v:TikTokVideo) WHERE v.video_content_embedding IS NOT NULL RETURN count(v) AS c'
    ),
    TIKTOK_SUMMARY_EMBED_IDX: (
        'MATCH (v:TikTokVideo) WHERE v.comment_summary_embedding IS NOT NULL RETURN count(v) AS c'
    ),
    'tiktok_comment_topic_embedding_index': (
        "MATCH (t:TikTokCommentTopic) WHERE t.embedding IS NOT NULL AND t.platform = 'tiktok' RETURN count(t) AS c"
    ),
}


def _indexed_node_counts() -> Dict[str, int]:
    out: Dict[str, int] = {}
    with driver.session(database=NEO4J_DATABASE) as s:
        for index_name, cy in _INDEXED_NODE_COUNTS_CYPHER.items():
            out[index_name] = int(s.run(cy).single()['c'])
    return out


def _k_for_index(index_name: str, counts: Dict[str, int]) -> int:
    if SIMILARITY_K_OVERRIDE_RAW:
        return max(SIMILARITY_K_MIN, int(SIMILARITY_K_OVERRIDE_RAW))
    n = counts.get(index_name, 0)
    k = int(SIMILARITY_K_FRACTION * n)
    return max(SIMILARITY_K_MIN, min(SIMILARITY_K_MAX, k))


QUERIES_FOR_THRESHOLD_TUNING = [
    'mental health',
    # 'grwm',
    # 'product review',
]

CONTENT_SCORES_CYPHER = f'''
CALL db.index.vector.queryNodes('{TIKTOK_CONTENT_EMBED_IDX}', $k, $q) YIELD node AS v, score
RETURN v.video_id AS video_id, v.video_title AS title, v.video_url AS url, score
'''

SUMMARY_SCORES_CYPHER = f'''
CALL db.index.vector.queryNodes('{TIKTOK_SUMMARY_EMBED_IDX}', $k, $q) YIELD node AS v, score
RETURN v.video_id AS video_id, v.video_title AS title, v.video_url AS url, score
'''

TOPIC_SCORES_CYPHER = '''
CALL db.index.vector.queryNodes('tiktok_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE t.platform = $platform
MATCH (v:TikTokVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE rel.platform = $platform
RETURN t.name AS topic_name,
       v.video_id AS video_id, v.video_title AS title, v.video_url AS url,
       score,
       rel.weight AS topic_weight,
       rel.position AS topic_position,
       score * coalesce(rel.weight, 1.0) AS weighted_similarity
'''

SCORE_BANDS: List[Tuple[str, float, float]] = [
    ('0.90–1.00', 0.9, 1.0000001),
    ('0.80–0.89', 0.8, 0.9),
    ('0.70–0.79', 0.7, 0.8),
    ('0.60–0.69', 0.6, 0.7),
    ('0.50–0.59', 0.5, 0.6),
]


def _fetch_rows(cypher: str, q: List[float], k: int) -> List[Dict[str, Any]]:
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(cypher, q=q, k=k)]


def _fetch_topic_score_rows(q: List[float], k: int) -> List[Dict[str, Any]]:
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(TOPIC_SCORES_CYPHER, q=q, k=k, platform=TIKTOK_PLATFORM)]


def _scores(rows: List[Dict[str, Any]]) -> List[float]:
    return [float(r['score']) for r in rows if r.get('score') is not None]


def _ascii_histogram(scores: List[float], n_bins: int = 20, width: int = 50) -> None:
    if not scores:
        print('  (no scores)')
        return
    lo, hi = min(scores), max(scores)
    lo_b, hi_b = 0.0, 1.0
    counts = [0] * n_bins
    under = over = 0
    for s in scores:
        if s < lo_b:
            under += 1
        elif s > hi_b:
            over += 1
        else:
            idx = int((s - lo_b) / (hi_b - lo_b) * n_bins)
            if idx >= n_bins:
                idx = n_bins - 1
            counts[idx] += 1
    mx = max(counts) if counts else 1
    print(f'  score min={lo:.4f} max={hi:.4f}  (binning [0,1] with {n_bins} bins; below0={under} above1={over})')
    for i, c in enumerate(counts):
        left = lo_b + (hi_b - lo_b) * i / n_bins
        right = lo_b + (hi_b - lo_b) * (i + 1) / n_bins
        bar = '#' * max(1, int(width * c / mx)) if c else ''
        print(f'  [{left:.2f},{right:.2f}): {c:4d}  {bar}')


def _band_examples(rows: List[Dict[str, Any]], label: str, lo: float, hi: float, n: int = 5) -> None:
    band = [r for r in rows if r.get('score') is not None and lo <= float(r['score']) < hi]
    band.sort(key=lambda r: float(r['score']), reverse=True)
    print(f'  band {label}: {len(band)} hits (showing top {min(n, len(band))})')
    for r in band[:n]:
        pprint(r)


def print_topic_retriever_topic_name_matches(
    rows: List[Dict[str, Any]],
    topic_name_substring: Optional[str] = None,
) -> None:
    if not topic_name_substring or not str(topic_name_substring).strip():
        return
    needle = str(topic_name_substring).strip().lower()
    matched = [r for r in rows if needle in (r.get('topic_name') or '').lower()]
    if not matched:
        return
    matched.sort(
        key=lambda r: float(
            r.get('weighted_similarity')
            if r.get('weighted_similarity') is not None
            else r.get('score') or 0.0
        ),
        reverse=True,
    )
    print('-' * 88)
    print(f'  TOPIC ROWS WITH topic_name containing {topic_name_substring.strip()!r}  (n={len(matched)})')
    for r in matched:
        pprint(r)


def analyze_similarity_for_query(
    query_name: str,
    counts: Dict[str, int],
    *,
    optional_topic_name_substring: Optional[str] = None,
) -> None:
    print('=' * 88)
    print(
        f'QUERY: {query_name!r}   (k ≈ {SIMILARITY_K_FRACTION * 100:.0f}% of indexed nodes per index, '
        f'min={SIMILARITY_K_MIN} max={SIMILARITY_K_MAX}'
        + (f", override={SIMILARITY_K_OVERRIDE_RAW!r}" if SIMILARITY_K_OVERRIDE_RAW else '')
        + ')'
    )
    print('=' * 88)
    q = embed_query(query_name)

    suites: List[Tuple[str, str]] = [
        (TIKTOK_CONTENT_EMBED_IDX, CONTENT_SCORES_CYPHER),
        (TIKTOK_SUMMARY_EMBED_IDX, SUMMARY_SCORES_CYPHER),
        ('tiktok_comment_topic_embedding_index', TOPIC_SCORES_CYPHER),
    ]

    for index_name, cypher in suites:
        k = _k_for_index(index_name, counts)
        if index_name == 'tiktok_comment_topic_embedding_index':
            rows = _fetch_topic_score_rows(q, k)
        else:
            rows = _fetch_rows(cypher, q, k)
        scores = _scores(rows)
        print('-' * 88)
        n_idx = counts.get(index_name, 0)
        print(f'INDEX: {index_name}   indexed_nodes={n_idx}   k_used={k}   rows_returned={len(rows)}')
        _ascii_histogram(scores)
        for band_label, lo, hi in SCORE_BANDS:
            _band_examples(rows, band_label, lo, hi, n=5)
        if index_name == 'tiktok_comment_topic_embedding_index' and optional_topic_name_substring:
            print_topic_retriever_topic_name_matches(rows, optional_topic_name_substring)
        print()


_COUNTS_GLOBAL = _indexed_node_counts()
print('Indexed node counts (used to size k per index):', _COUNTS_GLOBAL)

for _q in QUERIES_FOR_THRESHOLD_TUNING:
    analyze_similarity_for_query(_q, counts=_COUNTS_GLOBAL)


In [ ]:
_d = globals().get('driver')
if _d is not None:
    try:
        _d.close()
    except Exception:
        pass
    print('Closed Neo4j driver')
else:
    print('No Neo4j driver in memory - run the setup cell that creates `driver` first.')

_mc = globals().get('mongo_client')
if _mc is not None:
    try:
        _mc.close()
    except Exception:
        pass
    print('Closed Mongo client')
else:
    print('No Mongo client in memory - skipped.')
